# Volume 8 Drills — Offline RL & Imitation Learning

Work through each drill problem. Code cells are provided where applicable.

## Drill 1 — Offline RL: Bootstrapping Danger

**Question:** Why is bootstrapping (estimating targets from other estimates) particularly dangerous in **offline RL**?

**Answer (fill in):** ___

<details><summary>Answer</summary>

In online RL, when the agent queries Q(s', a') for action a' ∉ data, it can **explore** s' and collect real experience to correct errors.

In offline RL, the agent **cannot interact with the environment**. The dataset is fixed. When bootstrapping:

1. **Out-of-distribution (OOD) actions:** The Q-network may assign high values to actions not present in the dataset (no data to correct overestimates).
2. **Self-reinforcing overestimation:** Q(s, a) = r + γ max Q(s', a') → if Q(s', a') is overestimated for OOD a', this propagates backward and grows.
3. **Policy diverges:** The policy is updated to take OOD actions with inflated Q-values, leading to catastrophic performance.

This is called **distributional shift** or the **extrapolation error** problem. Offline RL methods (CQL, IQL, TD3+BC) specifically address it.
</details>

## Drill 2 — CQL: Preventing Overestimation

**Question:** How does **Conservative Q-Learning (CQL)** prevent Q-value overestimation in offline RL?

**Answer (fill in):** ___

<details><summary>Answer</summary>

CQL adds a **conservative regularization** term to the standard Bellman loss:

$$\mathcal{L}^{CQL}(Q) = \mathcal{L}^{Bellman}(Q) + \alpha \left( \mathbb{E}_{s \sim D,\ a \sim \mu}[Q(s,a)] - \mathbb{E}_{(s,a) \sim D}[Q(s,a)] \right)$$

- **First term:** Minimize Q-values for actions sampled from some distribution μ (often the current policy or uniform). This **pushes down** Q-values for OOD actions.
- **Second term:** Maximize Q-values for (s,a) pairs in the dataset. This **pushes up** Q-values for in-distribution actions.

**Effect:** CQL learns a **lower bound** on the true Q-function. The policy is conservative — it won't be lured by overestimated OOD Q-values.

The tradeoff: too large α → overly conservative, poor performance; too small α → insufficient protection.
</details>

## Drill 3 — Behavioral Cloning: Loss Function

**Question:** What is the loss function used in **Behavioral Cloning (BC)**?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Behavioral cloning is supervised learning on expert demonstrations `{(s_i, a_i^*)}`. The loss depends on the action space:

**Discrete actions (classification):**
$$\mathcal{L}_{BC} = -\frac{1}{N} \sum_{i=1}^N \log \pi(a_i^* | s_i; \theta) = \text{cross-entropy}(\pi(\cdot|s_i), a_i^*)$$

**Continuous actions (regression):**
$$\mathcal{L}_{BC} = \frac{1}{N} \sum_{i=1}^N \left\| \pi(s_i; \theta) - a_i^* \right\|^2 = \text{MSE}$$

BC is simple but suffers from **covariate shift** — it learns to mimic the expert distribution but cannot recover from states the expert never visited.
</details>

## Drill 4 — DAgger: Improving on Behavioral Cloning

**Question:** How does **DAgger (Dataset Aggregation)** address the main weakness of behavioral cloning?

**Answer (fill in):** ___

<details><summary>Answer</summary>

BC's weakness: it only trains on the **expert's state distribution**. When the learned policy makes small errors, it reaches states the expert never visited → no training data → cascading errors (**compounding / distribution shift**).

**DAgger fix:** Iteratively expand the training dataset with states visited by the **learned policy**, labeled by the expert:

1. Train initial policy π₁ on expert dataset D₀.
2. Roll out π_i in the environment to collect states {s}.
3. **Query the expert** for labels: collect {(s, a^*)} where a^* = expert(s).
4. Aggregate: D_{i+1} = D_i ∪ {(s, a^*)}.
5. Retrain policy on D_{i+1}. Repeat.

**Result:** The policy is trained on states it actually visits, not just expert states. DAgger provably achieves error O(T) vs BC's O(T²) in the horizon T. Requires **interactive access** to the expert (which may be expensive).
</details>

## Drill 5 — IRL: What Does It Recover?

**Question:** What does **Inverse Reinforcement Learning (IRL)** recover from expert demonstrations?

**Answer (fill in):** ___

<details><summary>Answer</summary>

IRL recovers the **reward function** R(s, a) (or R(s)) that explains the expert's behavior — i.e., the reward function for which the expert's demonstrations are optimal (or near-optimal).

**Motivation:** The true reward function may be:
- Unknown (e.g., human intent)
- Hard to specify manually
- More **transferable** than a policy (a reward function can be reused in new environments)

**Process:**
1. Observe expert demonstrations.
2. Infer R^* such that the expert is optimal under R^*.
3. Optionally: train a policy to optimize R^*.

IRL is **ill-posed** (many reward functions are consistent with any behavior), requiring regularization or maximum entropy assumptions (MaxEnt IRL).
</details>

## Drill 6 — GAIL: Discriminator Role

**Question:** In GAIL (Generative Adversarial Imitation Learning), what does the **discriminator** distinguish between?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The GAIL discriminator D(s, a) distinguishes between:
- **Expert transitions:** (s, a) pairs from expert demonstrations
- **Agent transitions:** (s, a) pairs generated by the current policy

It outputs a probability: D(s,a) → [0,1] where 1 = looks like expert, 0 = looks like agent.

**Training loop (GAN-style):**
1. **Discriminator update:** Maximize ability to classify expert vs agent.
   `L_D = -E_expert[log D(s,a)] - E_agent[log(1-D(s,a))]`
2. **Policy update:** Use `-log D(s,a)` (or `log D(s,a)`) as the reward signal — the agent is rewarded for **fooling the discriminator** into thinking it's the expert.

GAIL implicitly performs IRL + policy optimization simultaneously, and can match expert performance without access to the reward function.
</details>

## Drill 7 — RLHF: Reward Model Training Data

**Question:** In **RLHF (RL from Human Feedback)**, what is the reward model trained on?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The RLHF reward model is trained on **human preference pairs** (Bradley-Terry model):

1. Generate pairs of completions (y_A, y_B) from the same prompt x.
2. Human annotators indicate which is **preferred**: y_A ≻ y_B or y_B ≻ y_A.
3. The reward model r_φ is trained with cross-entropy loss:
   $$\mathcal{L} = -\mathbb{E}_{(x, y_w, y_l) \sim D}\left[\log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))\right]$$
   where y_w is the preferred ("winner") and y_l is the less-preferred ("loser") completion.

This reward model is then used to **fine-tune the language model** with PPO, maximizing:
$$\mathbb{E}[r_\phi(x, y)] - \beta \cdot D_{KL}(\pi_\theta || \pi_{ref})$$

The KL term prevents excessive divergence from the base model.
</details>

## Drill 8 — Code: Behavioral Cloning Loss

In [ ]:
import numpy as np

def softmax(logits):
    logits = np.array(logits, dtype=float)
    logits -= logits.max(axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=-1, keepdims=True)

def behavioral_cloning_loss(policy_logits, expert_actions):
    """
    Behavioral cloning cross-entropy loss for discrete actions.

    policy_logits: shape (N, n_actions)  -- raw network outputs
    expert_actions: shape (N,) -- integer action indices from expert
    Returns: scalar loss
    """
    probs = softmax(policy_logits)
    N = len(expert_actions)
    # Gather probability assigned to the expert action
    expert_probs = probs[np.arange(N), expert_actions]
    # Cross-entropy = -mean log probability of correct action
    loss = -np.mean(np.log(expert_probs + 1e-8))
    return loss

def bc_loss_mse(policy_actions, expert_actions):
    """Behavioral cloning MSE loss for continuous actions."""
    return np.mean((policy_actions - expert_actions) ** 2)

# Test: discrete BC loss
np.random.seed(0)
n_samples, n_actions = 8, 4
logits = np.random.randn(n_samples, n_actions)
expert_acts = np.array([1, 0, 3, 2, 1, 0, 2, 3])  # expert's choices

loss = behavioral_cloning_loss(logits, expert_acts)
print(f"BC cross-entropy loss: {loss:.4f}")
print(f"(Lower is better; max entropy baseline: {np.log(n_actions):.4f})")
print()

# Show per-sample probabilities assigned to expert actions
probs = softmax(logits)
print("Per-sample log-prob of expert action:")
for i, a in enumerate(expert_acts):
    print(f"  sample {i}: expert_a={a}, π(a|s)={probs[i,a]:.4f}, log={np.log(probs[i,a]):.4f}")

## Drill 9 — Distribution Shift: Compounding Errors in BC

**Question:** Explain how **distribution shift** causes behavioral cloning to fail through **compounding errors**.

**Answer (fill in):** ___

<details><summary>Answer</summary>

**The core problem:**
- BC is trained on states visited by the **expert** policy.
- At test time, the **learned policy** may take slightly different actions.
- These small differences cause the agent to visit **different states** than the expert ever encountered.
- The policy has **no training data** for these new states → makes poor decisions.
- Poor decisions lead to even **more unfamiliar states** → cascading errors.

**Formal bound (Ross & Bagnell 2010):**
If the policy makes ε error per step, after T steps:
- BC: total error ≈ **O(ε T²)** (quadratic in horizon)
- DAgger: total error ≈ **O(ε T)** (linear)

**Example:** A self-driving car trained only on smooth driving has never seen recovery from a near-miss. The first small deviation causes unfamiliar situations → more deviations → crash.
</details>

## Drill 10 — Decision Transformer: RL as Sequence Modeling

**Question:** How does the **Decision Transformer** formulate reinforcement learning as a sequence modeling problem?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The Decision Transformer reframes RL as **conditional sequence prediction** using a causal Transformer:

**Input sequence:**
$$\tau = (\hat{R}_1, s_1, a_1,\ \hat{R}_2, s_2, a_2,\ \ldots,\ \hat{R}_T, s_T, a_T)$$

Where $\hat{R}_t = \sum_{t'=t}^T r_{t'}$ is the **returns-to-go** (total future reward from t).

**At inference:**
- Specify a **desired return** $\hat{R}_1$ (e.g., maximum possible score)
- Provide current state s_t
- The Transformer generates the action a_t that would achieve that return

**Training:** Standard next-token prediction (cross-entropy for discrete, MSE for continuous actions) on offline trajectories — **no Bellman equations**, **no Q-learning**, **no policy gradients**.

**Key insight:** By conditioning on desired return, the model learns to "reach for" any target performance level, and high-return behavior can be elicited at inference without any online RL.
</details>